# USA Datafield Inspection — Template ↔ Category Recipes

**For students replicating the workshop.** This notebook runs entirely on the local CSVs in `datafields/USA/` — no Brain API calls, no auth required. Use it before mining to:

1. See *what's available* in USA delay-1 (MATRIX vs VECTOR, by category).
2. Pick a category × template combination that historically produces alphas.
3. Copy-paste the resulting config block into `pattern_search/config.py`.

Then go run `python pattern_search.py` and analyse with `query.py`.

## 0. Load the USA catalog

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
USA = PROJECT_DIR / "datafields" / "USA"

matrix = pd.read_csv(USA / "MATRIX.csv")
vector = pd.read_csv(USA / "VECTOR.csv")

print(f"MATRIX: {len(matrix):>5} fields")
print(f"VECTOR: {len(vector):>5} fields")
matrix.head(3)

## 1. What categories exist? Which are productive?

We'll rank categories by `alphaCount` median — the platform-wide median number of public alphas built per field. Higher = the community has found this category mineable.

In [ ]:
def category_summary(df, label):
    g = df.groupby("category_id").agg(
        fields=("id", "count"),
        coverage_med=("coverage", "median"),
        user_med=("userCount", "median"),
        alpha_med=("alphaCount", "median"),
        alpha_max=("alphaCount", "max"),
    ).round(3).sort_values("alpha_med", ascending=False)
    print(f"\n=== {label} ===")
    return g

category_summary(matrix, "MATRIX")

In [ ]:
category_summary(vector, "VECTOR")

**Reading the table** — `alpha_med` is the median "alphaCount" (community alphas built per field). The bigger this is, the more likely a fresh template lands a hit. `coverage_med` is the median fraction of stocks in TOP3000 that have non-NaN values; below ~0.4 means the field is sparse and templates need `ts_backfill` or `vec_avg`-style aggregation.

**Search-space size** — for a 2-placeholder pair template drawing both placeholders from one category:

In [ ]:
print("Pair-template search-space (N×N candidates) per MATRIX category:\n")
for cat, n in matrix["category_id"].value_counts().items():
    pairs = n * (n - 1) if n > 1 else 0
    print(f"  {cat:<14} {n:>5} fields  →  {pairs:>10,} pair candidates")

## 2. Template ↔ Category fitness matrix

Different template *shapes* suit different datafield *shapes*. Use this matrix as a starting heuristic; nothing replaces actually running a small batch and reading the check distribution.

| Template family | Shape it produces | Best categories | Why |
|---|---|---|---|
| **Pair ratio** `rank(ts_mean({a},10)/ts_mean({b},252))` | Cross-sectional, mean-reverting | `pv`, `fundamental` | Wants strictly-positive, smoothly-varying matrices. Negative or near-zero values blow up the ratio. |
| **Pair smoothed ratio** `ts_backfill(sqrt(rank(ts_mean({a},5)/ts_mean({b},252))),21)` | Same but tamer | `pv`, `fundamental`, `option` | `sqrt + ts_backfill` rescues low-coverage fields. |
| **Z-score blend** `rank(ts_decay_linear(-zscore(0.5*({a}+{b})),60))` | Smoothed momentum | `model`, `analyst`, `news` (MATRIX) | zscore handles unbounded raw signals; decay reduces turnover. |
| **Tanh saturator** `rank(ts_decay_linear(zscore({a})*tanh(3*(ts_rank({a},252)-0.5)),63))` | Single-placeholder, robust to outliers | `news`, `option`, `sentiment`, `socialmedia` | These categories have heavy tails / spikes — tanh kills outliers. |
| **trade_when (sparse)** `rank(ts_decay_linear(trade_when(ts_rank({a},252)>0.8,ts_rank({a},120),0),60))` | Sparse signal, low turnover | `news`, `sentiment`, `socialmedia`, `option` | Only trades on top-quintile signal — ideal for episodic data. |
| **VECTOR aggregator** `rank(ts_mean(vec_avg({a}),10)/ts_mean(vec_avg({b}),252))` | Pair ratio over vector arrays | `news` (VECTOR), `analyst` (VECTOR) | Required for any VECTOR datafield — `vec_avg`/`vec_sum` first reduces the vector to a scalar. |
| **Fundamental ts_rank** `ts_backfill(sqrt(rank(ts_mean({a},5)/ts_mean({a},252))),63)` | Single-placeholder fundamentals momentum | `fundamental` | Reuses the same field for short/long ratio — natural for slow-moving fundamentals. |

## 3. Recipes — copy-paste configs

Each recipe below is a complete drop-in for `pattern_search/config.py` — the three constants `EXPRESSION_TEMPLATE`, `REQUIRED_TYPE`, `PLACEHOLDER_CATEGORY`. Pick one, paste, run.

In [ ]:
RECIPES = {
    "pv_ratio": {
        "comment": "Classic price-volume pair ratio (43 fields → ~1800 pairs). Fast feedback.",
        "EXPRESSION_TEMPLATE": 'rank(ts_mean({a},10)/ts_mean({b},252))',
        "REQUIRED_TYPE":       {"a": "MATRIX", "b": "MATRIX"},
        "PLACEHOLDER_CATEGORY": {"a": "pv",     "b": "pv"},
    },
    "pv_smoothed": {
        "comment": "Smoother variant of pv_ratio — usually higher Sharpe pass-rate.",
        "EXPRESSION_TEMPLATE": 'ts_backfill(sqrt(rank(ts_mean({a},5)/ts_mean({b},252))),21)',
        "REQUIRED_TYPE":       {"a": "MATRIX", "b": "MATRIX"},
        "PLACEHOLDER_CATEGORY": {"a": "pv",     "b": "pv"},
    },
    "fundamental_pair": {
        "comment": "Fundamental MATRIX pair (318 fields → ~100k pairs — large search). Median 256 community alphas/field.",
        "EXPRESSION_TEMPLATE": 'ts_backfill(sqrt(rank(ts_mean({a},5)/ts_mean({b},252))),63)',
        "REQUIRED_TYPE":       {"a": "MATRIX",      "b": "MATRIX"},
        "PLACEHOLDER_CATEGORY": {"a": "fundamental", "b": "fundamental"},
    },
    "news_tanh": {
        "comment": "Single-placeholder news (MATRIX) with tanh saturator. Good for spiky news scores.",
        "EXPRESSION_TEMPLATE": 'rank(ts_decay_linear(zscore({a})*tanh(3*(ts_rank({a},252)-0.5)),63))',
        "REQUIRED_TYPE":       {"a": "MATRIX"},
        "PLACEHOLDER_CATEGORY": {"a": "news"},
    },
    "news_vector_pair": {
        "comment": "VECTOR news pair via vec_avg. 247 vector fields → ~60k pairs. Often the highest hit-rate.",
        "EXPRESSION_TEMPLATE": 'rank(ts_mean(vec_avg({a}),10)/ts_mean(vec_avg({b}),252))',
        "REQUIRED_TYPE":       {"a": "VECTOR", "b": "VECTOR"},
        "PLACEHOLDER_CATEGORY": {"a": "news",   "b": "news"},
    },
    "analyst_vector_pair": {
        "comment": "VECTOR analyst pair (184 fields → ~33k pairs). Earnings/revisions territory.",
        "EXPRESSION_TEMPLATE": 'rank(ts_mean(vec_avg({a}),10)/ts_mean(vec_avg({b}),252))',
        "REQUIRED_TYPE":       {"a": "VECTOR",  "b": "VECTOR"},
        "PLACEHOLDER_CATEGORY": {"a": "analyst", "b": "analyst"},
    },
    "sentiment_trade_when": {
        "comment": "Sparse trading on top-quintile sentiment. Low turnover, often passes LOW_TURNOVER but watch LOW_FITNESS.",
        "EXPRESSION_TEMPLATE": 'rank(ts_decay_linear(trade_when(ts_rank({a},252)>0.8,ts_rank({a},120),0),60))',
        "REQUIRED_TYPE":       {"a": "MATRIX"},
        "PLACEHOLDER_CATEGORY": {"a": "sentiment"},
    },
    "option_zscore": {
        "comment": "Option-derived MATRIX with zscore + decay. 138 fields, mid-coverage (~0.7).",
        "EXPRESSION_TEMPLATE": 'rank(ts_decay_linear(-zscore(0.5*({a}+{b})),60))',
        "REQUIRED_TYPE":       {"a": "MATRIX", "b": "MATRIX"},
        "PLACEHOLDER_CATEGORY": {"a": "option", "b": "option"},
    },
}

print("Available recipes:")
for name, r in RECIPES.items():
    print(f"  • {name:<22} — {r['comment']}")

## 4. Pick a recipe and validate it

Pick one recipe and the cell below will: (a) confirm the categories actually exist in the catalog, (b) compute the search-space size, (c) print the exact config snippet to paste into `pattern_search/config.py`.

In [ ]:
PICK = "pv_smoothed"   # change me

r = RECIPES[PICK]

# Resolve each placeholder to its bucket, count fields
buckets = {}
for ph, want_type in r["REQUIRED_TYPE"].items():
    cat = r["PLACEHOLDER_CATEGORY"][ph]
    df = matrix if want_type == "MATRIX" else vector
    fields = df[df["category_id"] == cat]["id"].tolist()
    buckets[ph] = (want_type, cat, fields)
    print(f"  {{{ph}}}  type={want_type:<6} cat={cat:<12} fields={len(fields)}")

# Search-space size (Cartesian, dedup by REQUIRE_DIFFERENT_SAME_CATEGORY=True)
sizes = [len(b[2]) for b in buckets.values()]
if len(sizes) == 1:
    total = sizes[0]
else:
    total = 1
    for s in sizes:
        total *= s
    # Same-category placeholders → exclude diagonal
    cats = [b[1] for b in buckets.values()]
    if len(set(cats)) == 1:
        total -= sizes[0]

print(f"\nSearch space: {total:,} candidate expressions")
print(f"\n--- paste this into pattern_search/config.py ---")
print(f'EXPRESSION_TEMPLATE = {r["EXPRESSION_TEMPLATE"]!r}')
print(f'REQUIRED_TYPE        = {r["REQUIRED_TYPE"]}')
print(f'PLACEHOLDER_CATEGORY = {r["PLACEHOLDER_CATEGORY"]}')

## 5. Sample the first few rendered expressions

What will the miner actually submit? Render the first 5 expressions for the chosen recipe — useful for sanity-checking syntax before launching.

In [ ]:
import itertools

names = list(buckets.keys())
lists = [buckets[n][2] for n in names]

shown = 0
for combo in itertools.product(*lists):
    # Skip diagonal when placeholders share category
    cats = [buckets[n][1] for n in names]
    if len(set(cats)) == 1 and len(set(combo)) != len(combo):
        continue
    expr = r["EXPRESSION_TEMPLATE"].format(**dict(zip(names, combo)))
    print(expr)
    shown += 1
    if shown >= 5:
        break

## 6. Browse a category's fields directly

Curious what's in a category? Filter and view descriptions / coverage / community alpha-counts.

In [ ]:
BROWSE_CAT = "pv"
BROWSE_TYPE = "MATRIX"   # "MATRIX" or "VECTOR"

df = matrix if BROWSE_TYPE == "MATRIX" else vector
view = (
    df[df["category_id"] == BROWSE_CAT]
    [["id", "description", "dataset_id", "coverage", "userCount", "alphaCount"]]
    .sort_values("alphaCount", ascending=False)
)
view.head(20)

## 7. After mining — interpret the check distribution

Once `python pattern_search.py` has produced ~50+ alphas, run this in `query.py` (or DuckDB directly) and use the table below to choose the next template.

```sql
SELECT c.name, c.result, count(*) AS n
FROM alphas, UNNEST("is".checks) AS t(c)
GROUP BY 1,2 ORDER BY 1,3 DESC;
```

| Dominant FAIL | Diagnosis | Switch to |
|---|---|---|
| `LOW_SHARPE` | Signal too noisy | Add `ts_decay_linear` / `zscore` / longer windows. Try `news_tanh` recipe. |
| `LOW_FITNESS` | Returns too small relative to volatility | Change category — `news` and `option` typically have higher info content than raw `pv`. |
| `LOW_TURNOVER` | Signal too sticky / always trading the same names | Shorten the windows (`ts_mean({a},5)` instead of `,252`), or remove `ts_backfill`. |
| `HIGH_TURNOVER` | Trading every day | Add a smoother (`ts_decay_linear(...,60)`) or use `trade_when` to sparsify. |
| `LOW_SUB_UNIVERSE_SHARPE` | Works only on the top names | Add `group_neutralize(..., subindustry)` and bump `neutralization='SUBINDUSTRY'`. |
| `MATCHES_COMPETITION` | Too similar to existing community alphas | Pick a less-mined category (e.g., `option`, `socialmedia`) or a different template family. |
| `SELF_CORRELATION` PENDING | Brain hasn't scored yet | Wait minutes/hours then re-fetch. Not a problem. |

## Cheat sheet for students

1. **Pick a recipe** — start with `pv_smoothed` (small + fast feedback).
2. **Paste the snippet** into `pattern_search/config.py`.
3. **Run** `python pattern_search.py` for ~5 minutes.
4. **Inspect** with `python query.py` and the queries in `README.md → Useful queries`.
5. **Read** the check distribution. Use the diagnostic table above to pick the next recipe.
6. **Iterate**. The miner skips already-simulated expressions, so changing template grows your dataset rather than overwriting it.